# Chapter 6: The Power of Averages

This notebook accompanies **Chapter 6** of the lecture notes.

**Agenda**

📐 · 🎯 · 📊 · 🎲 · 🥧 · 📉 · 🏁

**Next steps (take it from here):** 🌍 · ⚖️ · 🔄

> **Tip:** Run cells top to bottom. Later cells depend on earlier ones.

---

*Welcome back to the Coffee Lab.* Today we measure total caffeine extraction over time (integration), sample random coffee batches to estimate quality (Monte Carlo), and optimize a roast profile using noisy gradient estimates (SGD). Every method in this chapter boils down to one idea: **computing an average**.*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from checks import check_trapezoid, check_simpson, check_monte_carlo, check_pi, check_sgd_step


def tufte_axis(ax):
    """Remove spines, keep only outward ticks on left and bottom."""
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(axis='both', which='both', direction='out',
                   length=5, width=1.2, colors='black',
                   top=False, right=False)

## From Rectangles to Random Darts

Computing an integral is computing an average. Think of it this way: if the caffeine extraction rate changes over time, the total caffeine extracted is the integral of that rate — an average rate times the brewing duration.

The trapezoid rule averages function values on a regular grid, weighted by the geometry of trapezoids. Simpson's rule fits parabolas through triples of points. Monte Carlo integration throws random darts and averages what they hit — like sampling random coffee batches from a production line to estimate overall quality. Stochastic Gradient Descent applies the same random-averaging idea to gradients, letting us optimize a roast profile without tasting every single batch.

All exercises in this notebook use the same reference integral so you can compare methods head-to-head.

> Why would anyone use random sampling to compute an integral that a simple formula can handle exactly?

<details><summary>Thought</summary>

In one dimension, deterministic rules like Simpson's converge far faster than Monte Carlo. But in high dimensions, a regular grid with 10 points per axis needs 10 to the power d evaluations — a number that explodes past d = 5 or so. Monte Carlo needs the same N samples regardless of dimension, making it the only viable option when d is large. Imagine optimizing coffee flavor across dozens of parameters — only random sampling scales.
</details>

In [ ]:
# Reference integral: integral from 0 to 1 of e^x dx = e - 1
f = np.exp
a, b = 0, 1
exact = np.e - 1

print(f"Target: integral from {a} to {b} of e^x dx")
print(f"Exact value: {exact:.10f}")

### 📐 Composite Trapezoid Rule

> Imagine measuring caffeine concentration at regular time intervals during a brew. The trapezoid rule connects consecutive measurements with straight lines and sums the resulting trapezoid areas to estimate total extraction. Each interior measurement sits on the boundary of two adjacent trapezoids, so it gets counted twice. But the first and last measurements each belong to only one trapezoid. How does that asymmetry show up in the weight pattern?

<details><summary>Thought</summary>

Interior points are shared between the trapezoid on their left and the one on their right, so they appear in two area calculations and receive weight 2. The first and last points are each an edge of only one trapezoid, so they get weight 1. The full pattern is 1-2-2-...-2-1, all multiplied by h/2.
</details>

Implement the composite trapezoid rule with n subintervals. Revisit the lecture notes for the formula.

Useful operations: `np.linspace()`, `np.sum()`.

In [ ]:
def trapezoid(f, a, b, n):
    """Composite trapezoid rule with n subintervals."""
    h = (b - a) / n
    x = np.linspace(a, b, n + 1)
    result = f(x[0]) + f(x[-1])
    for i in range(1, n):
        result += 2 * f(x[i])
    return result * h / 2


check_trapezoid(trapezoid, f, a, b, 4)
check_trapezoid(trapezoid, f, a, b, 16)
check_trapezoid(trapezoid, f, a, b, 64)

### 🎯 Composite Simpson's Rule

> Simpson's rule fits a parabola through every triple of consecutive measurements rather than connecting them with straight lines. For a smooth extraction curve like e to the x, should fitting parabolas make a noticeable difference compared to straight-line segments, or is the curve already nearly linear on small time intervals?

<details><summary>Thought</summary>

Even on small subintervals, e to the x has nonzero curvature. The trapezoid rule ignores that curvature entirely, while Simpson's rule captures it through the parabolic fit. The payoff is dramatic: Simpson's error shrinks as h to the fourth power versus h squared for the trapezoid rule. Doubling n reduces Simpson's error by a factor of 16 instead of 4.
</details>

Implement the composite Simpson's 1/3 rule. The number of subintervals n must be even. Revisit the lecture notes for the weight pattern.

Useful operations: `np.linspace()`, modulo `%` for alternating weights.

In [ ]:
def simpson(f, a, b, n):
    """Composite Simpson's 1/3 rule with n subintervals (n must be even)."""
    h = (b - a) / n
    x = np.linspace(a, b, n + 1)
    result = f(x[0]) + f(x[-1])
    for i in range(1, n):
        if i % 2 == 1:
            result += 4 * f(x[i])
        else:
            result += 2 * f(x[i])
    return result * h / 3


check_simpson(simpson, f, a, b, 4)
check_simpson(simpson, f, a, b, 16)
check_simpson(simpson, f, a, b, 64)

### 📊 Convergence: Trapezoid vs. Simpson

> When you double the number of measurement points, the trapezoid error should shrink by roughly 4x and Simpson's error by roughly 16x. Run the cell below and check whether the ratios match the theoretical predictions. If a ratio is off, what might explain the discrepancy?

<details><summary>Thought</summary>

The theoretical ratios assume the function is smooth enough for the leading error term to dominate. For e to the x both rules behave as predicted. Very small n values may show slightly irregular ratios because higher-order error terms have not yet become negligible.
</details>

In [ ]:
n_values = [2, 4, 8, 16, 32, 64, 128]

print(f"{'n':>6} | {'Trapezoid':>14} | {'Trap Error':>12} | {'Trap Ratio':>10} | {'Simpson':>14} | {'Simp Error':>12} | {'Simp Ratio':>10}")
print(f"{'-'*90}")

prev_trap_err = None
prev_simp_err = None

for n in n_values:
    t_val = trapezoid(f, a, b, n)
    s_val = simpson(f, a, b, n) if n % 2 == 0 else None

    if t_val is not None:
        t_err = abs(t_val - exact)
        t_ratio = f"{prev_trap_err / t_err:10.2f}" if (prev_trap_err is not None and t_err > 0) else "         -"
        prev_trap_err = t_err
        t_str = f"{t_val:>14.10f}"
        te_str = f"{t_err:>12.2e}"
    else:
        t_str = "           N/A"
        te_str = "         N/A"
        t_ratio = "         -"

    if s_val is not None:
        s_err = abs(s_val - exact)
        s_ratio = f"{prev_simp_err / s_err:10.2f}" if (prev_simp_err is not None and s_err > 0) else "         -"
        prev_simp_err = s_err
        s_str = f"{s_val:>14.10f}"
        se_str = f"{s_err:>12.2e}"
    else:
        s_str = "           N/A"
        se_str = "         N/A"
        s_ratio = "         -"

    print(f"{n:6d} | {t_str} | {te_str} | {t_ratio} | {s_str} | {se_str} | {s_ratio}")

print(f"\nExpected ratios: Trapezoid ~4 (O(h^2)),  Simpson ~16 (O(h^4))")

### 🎲 Monte Carlo Integration

> Monte Carlo replaces a structured grid with random samples — like pulling random coffee batches off the production line and averaging their quality scores to estimate the overall batch quality. The error decreases as 1 over the square root of N regardless of dimension. For the one-dimensional integral we have been testing, Simpson with n = 8 already reaches an error below 1e-7. How many Monte Carlo samples would you need to match that accuracy?

<details><summary>Thought</summary>

Monte Carlo error scales as 1 over the square root of N. To get an error of 1e-7 you would need on the order of 1e14 samples — completely impractical. In one dimension, deterministic quadrature wins overwhelmingly. Monte Carlo only becomes competitive when the dimension is high enough that grid-based methods are impossible.
</details>

Implement Monte Carlo integration for a 1D function f on the interval [a, b] using n_samples random points. Revisit the lecture notes for the estimator formula.

Useful operations: `np.random.uniform()`, `np.mean()`.

In [ ]:
def monte_carlo_integrate(f, a, b, n_samples):
    """Monte Carlo estimate of the integral of f from a to b."""
    samples = np.random.uniform(a, b, n_samples)
    return (b - a) * np.mean(f(samples))


check_monte_carlo(monte_carlo_integrate, f, a, b, 10_000)
check_monte_carlo(monte_carlo_integrate, f, a, b, 100_000)

### 🥧 Estimate Pi by Throwing Darts

> A quarter circle of radius 1 sits inside the unit square [0, 1] x [0, 1]. Its area is pi/4. If you throw N darts uniformly at the square and count how many land inside the quarter circle, the fraction approximates pi/4. Think of it as randomly sampling points in a coffee roasting parameter space and checking which ones fall inside the "good roast" region. But the estimate is noisy — how does the noise behave as you increase N?

<details><summary>Thought</summary>

Each dart is a Bernoulli trial with probability pi/4 of landing inside. The fraction converges to pi/4 by the law of large numbers, and the standard error decreases as 1 over the square root of N. To halve the error you need to quadruple the number of darts — the same square-root scaling as Monte Carlo integration.
</details>

Implement the dart-throwing pi estimator. Sample x and y uniformly from [0, 1], check how many satisfy x squared plus y squared being at most 1, and scale the result.

Useful operations: `np.random.uniform()`, `np.sum()`, boolean indexing.

In [ ]:
def estimate_pi(n_samples):
    """Estimate pi using the dart-throwing Monte Carlo method."""
    x = np.random.uniform(0, 1, n_samples)
    y = np.random.uniform(0, 1, n_samples)
    inside = np.sum(x**2 + y**2 <= 1)
    return 4 * inside / n_samples


check_pi(estimate_pi, 10_000)
check_pi(estimate_pi, 100_000)

### 📉 SGD Step

> Stochastic Gradient Descent is Monte Carlo applied to gradients. Instead of averaging f(x) over random points to estimate an integral, SGD averages gradient samples to estimate the full gradient — like adjusting a roast profile based on feedback from a random subset of tasters rather than polling every customer. The update rule subtracts a scaled gradient from the current parameters. What happens if the learning rate is too large?

<details><summary>Thought</summary>

If the learning rate is too large, the step overshoots the optimal roast and the parameters oscillate or diverge. The gradient points downhill locally, but a large step can jump past the valley and land on the opposite slope. The learning rate controls the trade-off between fast progress and stable convergence.
</details>

Implement a single SGD update: given the current parameters theta, a gradient vector, and a learning rate, return the updated parameters. Revisit the lecture notes for the update rule.

Useful operations: basic arithmetic with NumPy arrays.

In [ ]:
def sgd_step(theta, grad, lr):
    """Return updated parameters after one SGD step."""
    return theta - lr * grad


check_sgd_step(sgd_step, np.array([5.0, 3.0]), np.array([2.0, -1.0]), 0.1)
check_sgd_step(sgd_step, np.array([0.0]), np.array([4.0]), 0.5)

### 🏁 Recap

**What we did:**
- 📐 Implemented the composite trapezoid rule — measuring total caffeine extraction by connecting time samples with straight lines — and saw O(h squared) convergence.
- 🎯 Implemented Simpson's rule — fitting parabolas for a more accurate extraction estimate — and saw O(h to the fourth) convergence.
- 📊 Compared convergence rates side by side in a table.
- 🎲 Built a Monte Carlo integrator — sampling random coffee batches to estimate overall quality — and observed the 1 over square root of N scaling.
- 🥧 Estimated pi by throwing random darts.
- 📉 Wrote the SGD update rule — optimizing a roast profile using noisy gradient estimates from random mini-batches of tasters.

**Key takeaways:**
- Deterministic quadrature (trapezoid, Simpson) converges fast in low dimensions but breaks down as dimension grows.
- Monte Carlo converges slowly (1 over square root of N) but its cost is independent of dimension — essential when optimizing coffee across many parameters.
- SGD is Monte Carlo integration of gradients: each mini-batch average is a random estimate of the true gradient.

**Now head back for self-check questions and key learnings in the lecture notes.**

## Take It from Here — Next Steps (Optional)

The exercises below are **optional** extensions. They deepen your intuition but are not required to follow the rest of the course. Work through them at your own pace after the session.

### 🌍 The Curse of Dimensionality

> With 10 grid points per dimension, how many total evaluations does a deterministic quadrature rule need in d dimensions? At what dimension does this become absurd? Think about optimizing coffee flavor across grind size, water temperature, extraction time, bean variety, roast level, milk ratio...

<details><summary>Thought</summary>

The grid has 10 to the power d points. At d = 10 that is 10 billion, and at d = 20 it exceeds the number of atoms in the observable universe. Meanwhile, Monte Carlo with N = 100,000 samples works just as well in d = 20 as in d = 1. This is why neural network training (where d can be millions) relies on stochastic methods.
</details>

In [ ]:
# Curse of dimensionality demonstration
N_per_dim = 10
dimensions = [1, 2, 3, 5, 10, 20, 50, 100]

print(f"{'Dimension d':>12} | {'Grid Points (N=10)':>22} | {'Status':>12}")
print(f"{'-'*52}")

for d in dimensions:
    points = N_per_dim ** d
    if d <= 3:
        status = "Easy"
    elif d <= 5:
        status = "Doable"
    elif d <= 10:
        status = "Hard"
    elif d <= 20:
        status = "Impossible"
    else:
        status = "Absurd"
    pts_str = f"{points:>22,}" if points < 1e12 else f"{points:>22.2e}"
    print(f"{d:>12d} | {pts_str} | {status:>12}")

print(f"\nMonte Carlo needs only N samples regardless of dimension.")
print(f"Its error is O(1/sqrt(N)) no matter what d is.")

### ⚖️ Monte Carlo vs. Simpson

> For the 1D integral of e to the x, compare the accuracy of Simpson with a small n to Monte Carlo with a large N. Which method would you choose for a single-parameter brewing optimization? When does Monte Carlo become the better choice?

<details><summary>Thought</summary>

Simpson with n = 8 (only 9 function evaluations) already has an error below 1e-7. Monte Carlo with 100,000 samples barely reaches error 1e-2 or 1e-3. In one dimension, deterministic rules are far superior. But in 20 dimensions, Simpson would need 9 to the 20 evaluations while Monte Carlo still needs just 100,000.
</details>

In [ ]:
# Side-by-side comparison: Simpson vs. Monte Carlo on the reference integral
print(f"{'Method':<28} {'N/n':>8} {'Approx':>14} {'Error':>12}")
print(f"{'-'*66}")

for n in [4, 8, 16, 64]:
    s = simpson(f, a, b, n)
    if s is not None:
        print(f"{'Simpson (n=' + str(n) + ')':<28} {n:>8d} {s:>14.10f} {abs(s - exact):>12.2e}")

print()
for N in [100, 1_000, 10_000, 100_000]:
    np.random.seed(42)
    mc = monte_carlo_integrate(f, a, b, N)
    if mc is not None:
        print(f"{'Monte Carlo (N=' + str(N) + ')':<28} {N:>8d} {mc:>14.10f} {abs(mc - exact):>12.2e}")

print(f"\nIn 1D, Simpson wins easily. But try d = 20 dimensions...")

### 🔄 Full SGD Loop

> A single SGD step adjusts the roast profile slightly based on one batch of taster feedback. A full training loop repeats this step many times, each time sampling a fresh mini-batch. How does the batch size affect the smoothness of convergence?

<details><summary>Thought</summary>

Smaller batches produce noisier gradient estimates (higher variance, since you average fewer tasters). The trajectory bounces around more but each step is cheap. Larger batches give smoother trajectories but cost more per step. The variance of the mini-batch gradient scales as 1 over the batch size — the same 1 over N scaling from Monte Carlo.
</details>

In [ ]:
# Full SGD loop on a simple problem: minimize L(theta) = mean((data - theta)^2)
# The optimal theta is the mean of the data.

np.random.seed(42)
data = np.array([0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0])
optimal = np.mean(data)
print(f"Data: {data}")
print(f"Optimal theta (mean): {optimal:.4f}")

# SGD with different batch sizes
lr = 0.1
n_steps = 50
batch_sizes = [1, 2, 4, 7]

fig, axes = plt.subplots(1, len(batch_sizes), figsize=(14, 3.5), sharey=True)

for ax, bs in zip(axes, batch_sizes):
    theta = 0.0
    history = [theta]

    for step in range(n_steps):
        # Sample a mini-batch
        batch = data[np.random.choice(len(data), size=bs, replace=False)]
        # Mini-batch gradient of MSE: 2 * (theta - mean(batch))
        grad = 2 * (theta - np.mean(batch))
        # Use your sgd_step function
        result = sgd_step(np.array([theta]), np.array([grad]), lr)
        if result is not None:
            theta = float(result[0])
        history.append(theta)

    label = "full batch" if bs == len(data) else f"batch={bs}"
    ax.plot(history, color='tab:blue', linewidth=0.8)
    ax.axhline(optimal, color='tab:red', linewidth=0.6, linestyle='--')
    ax.set_title(label, fontsize=10)
    ax.set_xlabel('Step')
    if ax == axes[0]:
        ax.set_ylabel('theta')
    tufte_axis(ax)

plt.tight_layout()
plt.show()

print(f"\nSmaller batch = noisier trajectory but cheaper steps.")
print(f"Larger batch = smoother convergence but more computation per step.")
print(f"Variance of mini-batch gradient ~ 1/batch_size (same 1/N from Monte Carlo).")